# 🧠 Model Training — Grid Search MLP · RNN · LSTM
### Deep Learning Project | Bitcoin Volatility Forecasting

**Estrategia:** Grid search sobre fold 0, lag=14d para seleccionar arquitectura por tipo.  
La ganadora se evalúa en los **5 folds completos** con las métricas del enunciado.

| Componente | Detalle |
|---|---|
| Tipos | MLP · RNN · LSTM |
| Neuronas/capa | 32 · 64 · 128 · 256 |
| Máx. capas | 3 |
| Dropout | on/off por capa — rate=0.3 |
| Selección | `gap_ratio < 0.15`, ordenado por `best_val_rmse` |
| Evaluación final | 5 folds × 4 lags × métricas + BDS test |


## 0. Verificación de GPU

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print()

devices = tf.config.list_physical_devices()
print("Dispositivos detectados:")
for d in devices:
    print(f"  {d}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU ACTIVA — {len(gpus)} dispositivo(s):")
    for g in gpus:
        print(f"   {g.name}")
else:
    print("\n⚠️  Sin GPU detectada — corriendo en CPU")
    print("   Si tienes DirectML instalado, verifica que el kernel sea 'Deep Learning (GPU)'")
    print("   El grid search tardará varias horas en CPU.")
    respuesta = input("\n¿Deseas continuar de todas formas? (s/n): ")
    if respuesta.lower() != 's':
        raise SystemExit("Ejecución cancelada. Configura la GPU antes de continuar.")


TensorFlow version: 2.10.0

Dispositivos detectados:
  PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
  PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')

✅ GPU ACTIVA — 2 dispositivo(s):
   /physical_device:GPU:0
   /physical_device:GPU:1


In [2]:
import pickle, warnings, itertools
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from scipy import stats

tf.random.set_seed(42)
np.random.seed(42)

os.makedirs('results', exist_ok=True)
os.makedirs('app', exist_ok=True)

print("✅ Imports completos")


✅ Imports completos


## 1. Cargar features.pkl

In [3]:
with open('features.pkl', 'rb') as f:
    data = pickle.load(f)

splits           = data['splits']
LAGS_LIST        = data['lags_list']
N_STEPS_FORECAST = data['n_steps_forecast']
TARGET_COL       = data['target_col']
time_series      = data['time_series']
dates            = data['dates']

GRID_LAG  = 14
GRID_FOLD = 0

print(f"Lags     : {LAGS_LIST}")
print(f"Horizonte: {N_STEPS_FORECAST} dias")
print(f"Grid search: lag={GRID_LAG}d, fold={GRID_FOLD}")


Lags     : [7, 14, 21, 28]
Horizonte: 7 dias
Grid search: lag=14d, fold=0


## 2. Metricas y BDS test

In [4]:
def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100

def mae(y_true, y_pred):  return np.mean(np.abs(y_true - y_pred))
def rmse(y_true, y_pred): return np.sqrt(np.mean((y_true-y_pred)**2))
def mse(y_true, y_pred):  return np.mean((y_true-y_pred)**2)

def metrics_per_horizon(y_true, y_pred):
    H = y_true.shape[1]
    result = {}
    for h in range(H):
        yt, yp = y_true[:,h], y_pred[:,h]
        result[f'h{h+1}'] = {
            'MAPE': round(mape(yt,yp),4), 'MAE':  round(mae(yt,yp),6),
            'RMSE': round(rmse(yt,yp),6), 'MSE':  round(mse(yt,yp),8),
        }
    result['mean'] = {
        k: round(np.mean([result[f'h{h+1}'][k] for h in range(H)]),6)
        for k in ['MAPE','MAE','RMSE','MSE']
    }
    return result

def bds_test(residuals, max_dim=2, eps_scale=0.7):
    x = np.asarray(residuals, dtype=float); x = x[~np.isnan(x)]
    n = len(x); eps = eps_scale * np.std(x)
    def c1_fn(arr):
        cnt = 0
        for i in range(len(arr)-1): cnt += np.sum(np.abs(arr[i+1:]-arr[i])<=eps)
        return 2*cnt/(n*(n-1))
    c1 = c1_fn(x); results = {}
    for m in range(2, max_dim+1):
        em = np.column_stack([x[i:n-m+i+1] for i in range(m)]); nr = len(em)
        cnt_m = 0
        for i in range(nr-1): cnt_m += np.sum(np.max(np.abs(em[i+1:]-em[i]),axis=1)<=eps)
        cm = 2*cnt_m/(nr*(nr-1))
        k_sum = sum(c1**(2*(m-j)) for j in range(1,m))
        var = (4/n)*(k_sum+(m-1)**2*c1**(2*m))
        stat = (cm-c1**m)/np.sqrt(max(var,1e-12))
        results[m] = {'statistic': round(float(stat),4),
                      'p_value': round(float(2*(1-stats.norm.cdf(abs(stat)))),4)}
    return results

print("✅ Metricas y BDS listos")


✅ Metricas y BDS listos


## 3. Configuracion del grid search

In [5]:
neuronas     = [32, 64, 128, 256]
max_capas    = 3
DROPOUT_RATE = 0.3
EPOCHS       = 50
BATCH_SIZE   = 32
PATIENCE     = 10

arquitecturas = []
for n_capas in range(1, max_capas+1):
    for comb in itertools.product(neuronas, repeat=n_capas):
        arquitecturas.append(comb)

def dropout_configs(n_capas):
    for mask in range(2**n_capas):
        yield tuple(bool((mask>>i)&1) for i in range(n_capas))

total_configs = sum(2**len(a) for a in arquitecturas)
print(f"Arquitecturas unicas : {len(arquitecturas)}")
print(f"Configs arch+dropout : {total_configs}")
print(f"Total grid (3 tipos) : {total_configs*3} entrenamientos")


Arquitecturas unicas : 84
Configs arch+dropout : 584
Total grid (3 tipos) : 1752 entrenamientos


## 4. Constructores — MLP · RNN · LSTM

In [6]:
def build_mlp(arch, dmask, input_dim, output_dim):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(input_dim,)))
    for idx, units in enumerate(arch):
        model.add(tf.keras.layers.Dense(units, activation='relu'))
        if dmask[idx]:
            model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.Dense(output_dim))
    model.compile(optimizer='adam', loss='mse',
                  metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    return model

def build_rnn(arch, dmask, timesteps, output_dim):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(timesteps, 1)))
    for idx, units in enumerate(arch[:-1]):
        model.add(tf.keras.layers.SimpleRNN(units, return_sequences=True))
        if dmask[idx]:
            model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.SimpleRNN(arch[-1], return_sequences=False))
    if dmask[-1]:
        model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.Dense(output_dim))
    model.compile(optimizer='adam', loss='mse',
                  metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    return model

def build_lstm(arch, dmask, timesteps, output_dim):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(timesteps, 1)))
    for idx, units in enumerate(arch[:-1]):
        model.add(tf.keras.layers.LSTM(units, return_sequences=True))
        if dmask[idx]:
            model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.LSTM(arch[-1], return_sequences=False))
    if dmask[-1]:
        model.add(tf.keras.layers.Dropout(DROPOUT_RATE))
    model.add(tf.keras.layers.Dense(output_dim))
    model.compile(optimizer='adam', loss='mse',
                  metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')])
    return model

print("✅ Constructores listos: MLP / RNN / LSTM")


✅ Constructores listos: MLP / RNN / LSTM


## 5. Funcion de grid search

In [7]:
import time, gc

def run_grid_search(model_type, X_tr, y_tr, X_val, y_val, input_lag, output_dim,
                    checkpoint_file=None, checkpoint_key=None,
                    partial_results=None, checkpoint_every=10):
    """Grid search con reanudacion y checkpoint incremental atomico.

    - partial_results: lista de dicts ya evaluados. Se reanuda en len(partial_results).
    - checkpoint_file / checkpoint_key: si se proveen, persiste cada checkpoint_every.
    - checkpoint_every: cada cuantas configs reportar progreso y guardar checkpoint.
    """
    # Lista plana de configs: se indexa por posicion para reanudar facil.
    all_configs = [(arch, dmask)
                   for arch in arquitecturas
                   for dmask in dropout_configs(len(arch))]
    n_total = len(all_configs)

    resultados = list(partial_results) if partial_results else []
    start_idx  = len(resultados)

    if start_idx > 0:
        print(f"  [{model_type}] Reanudando desde config {start_idx}/{n_total} "
              f"({n_total - start_idx} restantes)")
    elif start_idx == n_total:
        print(f"  [{model_type}] Ya tenia {n_total}/{n_total} — nada que correr.")
        return pd.DataFrame(resultados)

    es = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0)

    if model_type in ('RNN', 'LSTM'):
        X_tr_in  = X_tr.reshape(-1, input_lag, 1)
        X_val_in = X_val.reshape(-1, input_lag, 1)
    else:
        X_tr_in, X_val_in = X_tr, X_val

    block_start = time.time()

    for i in range(start_idx, n_total):
        arch, dmask = all_configs[i]

        tf.keras.backend.clear_session()
        tf.random.set_seed(42)

        if   model_type == 'MLP':  model = build_mlp(arch, dmask, input_lag, output_dim)
        elif model_type == 'RNN':  model = build_rnn(arch, dmask, input_lag, output_dim)
        else:                      model = build_lstm(arch, dmask, input_lag, output_dim)

        history = model.fit(
            X_tr_in, y_tr,
            validation_data=(X_val_in, y_val),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=[es], verbose=0)

        train_rmse = history.history['rmse']
        val_rmse   = history.history['val_rmse']
        best_val   = float(np.min(val_rmse))
        last_train = float(train_rmse[-1])
        gap        = abs(last_train - best_val)
        gap_ratio  = gap / best_val if best_val > 0 else 999.0

        resultados.append({
            'model_type':      model_type,
            'arquitectura':    arch,
            'dropout_mask':    dmask,
            'best_val_rmse':   round(best_val, 5),
            'last_train_rmse': round(last_train, 5),
            'gap':             round(gap, 5),
            'gap_ratio':       round(gap_ratio, 4),
            'epochs_run':      len(train_rmse),
            'train_curve':     train_rmse,
            'val_curve':       val_rmse,
        })

        # Contra el memory leak de TF 2.10 + DirectML en loops largos de modelos
        del model, history

        done = i + 1

        if done % checkpoint_every == 0 or done == n_total:
            elapsed = time.time() - block_start
            n_block = checkpoint_every if done % checkpoint_every == 0 \
                      else (done - (done // checkpoint_every) * checkpoint_every)
            n_block = n_block or checkpoint_every
            print(f"  [{model_type}] {done}/{n_total} | "
                  f"+{n_block} configs en {elapsed:.1f}s "
                  f"({elapsed/n_block:.2f}s/config)")

            if checkpoint_file is not None and checkpoint_key is not None:
                # Releer el checkpoint actual para no perder otras claves
                current = {}
                if os.path.exists(checkpoint_file):
                    try:
                        with open(checkpoint_file, 'rb') as f:
                            current = pickle.load(f)
                    except Exception:
                        current = {}
                current[checkpoint_key] = {
                    'results':  resultados,
                    'complete': False,
                }
                # Escritura atomica: si crashea a mitad del dump, el archivo
                # final nunca queda corrupto porque os.replace es atomico.
                tmp = checkpoint_file + '.tmp'
                with open(tmp, 'wb') as f:
                    pickle.dump(current, f)
                os.replace(tmp, checkpoint_file)

            gc.collect()
            block_start = time.time()

    return pd.DataFrame(resultados)

print("\u2705 Funcion grid search lista (con reanudacion + checkpoint atomico)")


✅ Funcion grid search lista (con reanudacion + checkpoint atomico)


## 6. Preparar datos — fold 0, lag=14d

In [8]:
sp = splits[GRID_LAG]

X_tr_raw  = sp['X'][GRID_FOLD];    y_tr_raw  = sp['y'][GRID_FOLD]
X_val_raw = sp['Xcv'][GRID_FOLD];  y_val_raw = sp['ycv'][GRID_FOLD]

sx = StandardScaler().fit(X_tr_raw)
sy = StandardScaler().fit(y_tr_raw)

X_tr_s  = sx.transform(X_tr_raw)
X_val_s = sx.transform(X_val_raw)
y_tr_s  = sy.transform(y_tr_raw)
y_val_s = sy.transform(y_val_raw)

print(f"Train : {X_tr_s.shape} -> {y_tr_s.shape}")
print(f"Val   : {X_val_s.shape} -> {y_val_s.shape}")


Train : (175, 14) -> (175, 7)
Val   : (58, 14) -> (58, 7)


## 7. Grid search con checkpoints

In [9]:
import time

CHECKPOINT_FILE  = 'grid_results_checkpoint.pkl'
CHECKPOINT_EVERY = 10

def _save_cp_atomic(cp):
    tmp = CHECKPOINT_FILE + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(cp, f)
    os.replace(tmp, CHECKPOINT_FILE)

def _is_complete(cp, mtype):
    if mtype not in cp:
        return False
    v = cp[mtype]
    if isinstance(v, dict):
        return v.get('complete', False)
    # DataFrame legacy del checkpoint anterior: se considera completo
    if isinstance(v, pd.DataFrame):
        return True
    return False

def _get_partial(cp, mtype):
    if mtype not in cp:
        return None
    v = cp[mtype]
    if isinstance(v, dict) and not v.get('complete', False):
        return v.get('results', [])
    return None

# --- Cargar checkpoint existente y reportar estado
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'rb') as f:
        checkpoint = pickle.load(f)
    print("\u2705 Checkpoint cargado. Estado por modelo:")
    for m in ['MLP', 'RNN', 'LSTM']:
        if m not in checkpoint:
            print(f"   {m:4s}: no existe")
        else:
            v = checkpoint[m]
            if isinstance(v, dict):
                n  = len(v.get('results', []))
                st = 'COMPLETO' if v.get('complete') else 'PARCIAL'
                print(f"   {m:4s}: {st} ({n} configs)")
            else:
                print(f"   {m:4s}: legacy DataFrame ({len(v)} filas) \u2192 completo")
else:
    checkpoint = {}
    print("No hay checkpoint. Empezando desde cero.")

# --- MLP: hardcodeado del run anterior (NO se reentrena)
if not _is_complete(checkpoint, 'MLP'):
    print("\nMLP \u2014 cargando resultado conocido del run anterior...")
    mlp_row = {
        'model_type':      'MLP',
        'arquitectura':    (256, 128, 128),
        'dropout_mask':    (False, False, False),
        'best_val_rmse':   0.56112,
        'last_train_rmse': 0.52118,
        'gap':             0.03994,
        'gap_ratio':       0.071,
        'epochs_run':      50,
    }
    checkpoint['MLP'] = {'results': [mlp_row], 'complete': True}
    _save_cp_atomic(checkpoint)
    pd.DataFrame([mlp_row]).to_csv('results/grid_MLP.csv', index=False)
    print("  MLP cargado: arch=(256,128,128) | val_rmse=0.56112 | gap_ratio=0.071")
    print("  CSV: results/grid_MLP.csv")
else:
    print("\nMLP ya completo \u2014 skipping.")

# --- RNN y LSTM con reanudacion automatica
for mtype in ['RNN', 'LSTM']:
    if _is_complete(checkpoint, mtype):
        print(f"\n{mtype} ya completo \u2014 skipping.")
        continue

    partial = _get_partial(checkpoint, mtype)
    print(f"\n{'='*60}")
    if partial:
        print(f"  GRID SEARCH \u2014 {mtype}  (reanudando desde {len(partial)}/584)")
    else:
        print(f"  GRID SEARCH \u2014 {mtype}  (empezando desde 0/584)")
    print(f"{'='*60}")

    t0 = time.time()
    if mtype == 'LSTM':
        print("  [device=/CPU:0] DirectML no implementa CudnnRNN; LSTM corre en CPU.")
        with tf.device('/CPU:0'):
            df = run_grid_search(
                mtype, X_tr_s, y_tr_s, X_val_s, y_val_s,
                input_lag=GRID_LAG, output_dim=N_STEPS_FORECAST,
                checkpoint_file=CHECKPOINT_FILE, checkpoint_key=mtype,
                partial_results=partial, checkpoint_every=CHECKPOINT_EVERY,
            )
    else:
        df = run_grid_search(
            mtype, X_tr_s, y_tr_s, X_val_s, y_val_s,
            input_lag=GRID_LAG, output_dim=N_STEPS_FORECAST,
            checkpoint_file=CHECKPOINT_FILE, checkpoint_key=mtype,
            partial_results=partial, checkpoint_every=CHECKPOINT_EVERY,
        )
    elapsed = time.time() - t0

    # Marcar completo y persistir
    checkpoint[mtype] = {'results': df.to_dict('records'), 'complete': True}
    _save_cp_atomic(checkpoint)

    # CSV de respaldo (sin curvas para que sea legible en Notion/Excel)
    df_csv = df.drop(columns=['train_curve', 'val_curve'], errors='ignore')
    df_csv.to_csv(f'results/grid_{mtype}.csv', index=False)

    candidatos = df[df['gap_ratio'] < 0.15].sort_values('best_val_rmse')
    best = candidatos.iloc[0] if len(candidatos) > 0 \
           else df.sort_values('best_val_rmse').iloc[0]
    print(f"  \u2705 {mtype} completo en {elapsed/60:.1f} min \u2014 CSV: results/grid_{mtype}.csv")
    print(f"  Mejor {mtype}: arch={best['arquitectura']} | "
          f"dropout={best['dropout_mask']} | "
          f"val_rmse={best['best_val_rmse']:.5f} | gap_ratio={best['gap_ratio']:.3f}")

# --- Reconstruir grid_results como DataFrames (para seccion 8 siguiente)
grid_results = {}
for mtype in ['MLP', 'RNN', 'LSTM']:
    v = checkpoint[mtype]
    grid_results[mtype] = pd.DataFrame(v['results']) if isinstance(v, dict) else v

print("\n\u2705 Grid search completo para los 3 modelos")


✅ Checkpoint cargado. Estado por modelo:
   MLP : legacy DataFrame (1 filas) → completo
   RNN : COMPLETO (584 configs)
   LSTM: PARCIAL (470 configs)

MLP ya completo — skipping.

RNN ya completo — skipping.

  GRID SEARCH — LSTM  (reanudando desde 470/584)
  [device=/CPU:0] DirectML no implementa CudnnRNN; LSTM corre en CPU.
  [LSTM] Reanudando desde config 470/584 (114 restantes)
  [LSTM] 480/584 | +10 configs en 310.1s (31.01s/config)
  [LSTM] 490/584 | +10 configs en 555.8s (55.58s/config)
  [LSTM] 500/584 | +10 configs en 404.7s (40.47s/config)
  [LSTM] 510/584 | +10 configs en 445.5s (44.55s/config)
  [LSTM] 520/584 | +10 configs en 549.7s (54.97s/config)
  [LSTM] 530/584 | +10 configs en 512.2s (51.22s/config)
  [LSTM] 540/584 | +10 configs en 413.6s (41.36s/config)
  [LSTM] 550/584 | +10 configs en 391.5s (39.15s/config)
  [LSTM] 560/584 | +10 configs en 402.0s (40.20s/config)
  [LSTM] 570/584 | +10 configs en 457.8s (45.78s/config)
  [LSTM] 580/584 | +10 configs en 534.7s (53

## 8. Seleccion automatica de arquitecturas

In [10]:
best_configs = {}

print("Arquitecturas seleccionadas:")
for mtype in ['MLP', 'RNN', 'LSTM']:
    df = grid_results[mtype]
    if 'gap_ratio' in df.columns:
        candidatos = df[df['gap_ratio'] < 0.15].sort_values('best_val_rmse')
        row = candidatos.iloc[0] if len(candidatos) > 0 else df.sort_values('best_val_rmse').iloc[0]
    else:
        row = df.iloc[0]
    best_configs[mtype] = {
        'arquitectura': row['arquitectura'],
        'dropout_mask': row['dropout_mask'],
        'best_val_rmse': row['best_val_rmse'],
        'gap_ratio': row['gap_ratio'],
    }
    print(f"  {mtype:4s} -> arch={row['arquitectura']} | "
          f"dropout={row['dropout_mask']} | "
          f"val_rmse={row['best_val_rmse']:.5f} | gap_ratio={row['gap_ratio']:.3f}")


Arquitecturas seleccionadas:
  MLP  -> arch=(256, 128, 128) | dropout=(False, False, False) | val_rmse=0.56112 | gap_ratio=0.071
  RNN  -> arch=(64, 32, 64) | dropout=(True, False, False) | val_rmse=0.58023 | gap_ratio=0.148
  LSTM -> arch=(32,) | dropout=(True,) | val_rmse=0.56483 | gap_ratio=0.444


## 9. Evaluacion completa — 5 folds × 4 lags × 3 modelos

In [12]:
import contextlib

results_all = {}

for mtype in ['MLP', 'RNN', 'LSTM']:
    arch   = best_configs[mtype]['arquitectura']
    dmask  = best_configs[mtype]['dropout_mask']
    _device = '/CPU:0' if mtype == 'LSTM' else None
    results_all[mtype] = {}
    print(f"\n{'='*60}\n  {mtype} — arch={arch} | dropout={dmask}\n{'='*60}")

    for lag in LAGS_LIST:
        sp = splits[lag]
        folds = list(sp['X'].keys())
        results_all[mtype][lag] = {}

        for fold in folds:
            X_tr_r  = sp['X'][fold];    y_tr_r  = sp['y'][fold]
            X_val_r = sp['Xcv'][fold];  y_val_r = sp['ycv'][fold]
            X_te_r  = sp['Xtest'][fold];y_te_r  = sp['ytest'][fold]

            sx = StandardScaler().fit(X_tr_r)
            sy = StandardScaler().fit(y_tr_r)

            X_tr_s  = sx.transform(X_tr_r);  X_val_s = sx.transform(X_val_r)
            X_te_s  = sx.transform(X_te_r);  y_tr_s  = sy.transform(y_tr_r)
            y_val_s = sy.transform(y_val_r)

            if mtype in ('RNN', 'LSTM'):
                X_tr_in  = X_tr_s.reshape(-1, lag, 1)
                X_val_in = X_val_s.reshape(-1, lag, 1)
                X_te_in  = X_te_s.reshape(-1, lag, 1)
            else:
                X_tr_in, X_val_in, X_te_in = X_tr_s, X_val_s, X_te_s

            tf.keras.backend.clear_session(); tf.random.set_seed(42)
            if   mtype == 'MLP':  model = build_mlp(arch, dmask, lag, N_STEPS_FORECAST)
            elif mtype == 'RNN':  model = build_rnn(arch, dmask, lag, N_STEPS_FORECAST)
            else:                 model = build_lstm(arch, dmask, lag, N_STEPS_FORECAST)

            es = tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0)

            ctx = tf.device(_device) if _device else contextlib.nullcontext()
            with ctx:
                model.fit(X_tr_in, y_tr_s, validation_data=(X_val_in, y_val_s),
                          epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[es], verbose=0)

                yhat_tr  = sy.inverse_transform(model.predict(X_tr_in,  verbose=0))
                yhat_val = sy.inverse_transform(model.predict(X_val_in, verbose=0))
                yhat_te  = sy.inverse_transform(model.predict(X_te_in,  verbose=0))

            m_test   = metrics_per_horizon(y_te_r, yhat_te)
            resid_h1 = y_te_r[:,0] - yhat_te[:,0]
            bds_res  = bds_test(resid_h1, max_dim=2)
            bds_pval = bds_res[2]['p_value']

            print(f"  lag={lag}d fold={fold} | RMSE={m_test['mean']['RMSE']:.4f} | "
                  f"MAPE={m_test['mean']['MAPE']:.2f}% | "
                  f"BDS p={bds_pval:.3f} {'OK' if bds_pval>0.05 else 'WARN'}")

            results_all[mtype][lag][fold] = {
                'metrics_test': m_test,
                'rmse_test':    m_test['mean']['RMSE'],
                'bds_pval':     bds_pval,
                'yhat_train':   yhat_tr,
                'yhat_val':     yhat_val,
                'yhat_test':    yhat_te,
                'y_train_raw':  y_tr_r,
                'y_val_raw':    y_val_r,
                'y_test_raw':   y_te_r,
                'resid_h1':     resid_h1,
            }

# Guardar checkpoint de resultados finales
with open('results_all_checkpoint.pkl', 'wb') as f:
    pickle.dump(results_all, f)
print("\n✅ Evaluacion completa — checkpoint guardado en results_all_checkpoint.pkl")


  MLP — arch=(256, 128, 128) | dropout=(False, False, False)
  lag=7d fold=0 | RMSE=0.2508 | MAPE=40.93% | BDS p=0.912 OK
  lag=7d fold=1 | RMSE=0.2558 | MAPE=42.34% | BDS p=0.966 OK
  lag=7d fold=2 | RMSE=0.2435 | MAPE=40.85% | BDS p=0.998 OK
  lag=7d fold=3 | RMSE=0.2267 | MAPE=40.87% | BDS p=0.943 OK
  lag=7d fold=4 | RMSE=0.2646 | MAPE=44.02% | BDS p=0.977 OK
  lag=14d fold=0 | RMSE=0.1971 | MAPE=38.52% | BDS p=0.971 OK
  lag=14d fold=1 | RMSE=0.2135 | MAPE=44.76% | BDS p=0.909 OK
  lag=14d fold=2 | RMSE=0.2209 | MAPE=49.45% | BDS p=0.974 OK
  lag=14d fold=3 | RMSE=0.2135 | MAPE=48.10% | BDS p=0.992 OK
  lag=14d fold=4 | RMSE=0.2773 | MAPE=45.89% | BDS p=0.977 OK
  lag=21d fold=0 | RMSE=0.2325 | MAPE=44.24% | BDS p=0.936 OK
  lag=21d fold=1 | RMSE=0.2349 | MAPE=42.49% | BDS p=0.979 OK
  lag=21d fold=2 | RMSE=0.2495 | MAPE=41.29% | BDS p=0.863 OK
  lag=21d fold=3 | RMSE=0.2513 | MAPE=38.86% | BDS p=0.996 OK
  lag=21d fold=4 | RMSE=0.1934 | MAPE=40.05% | BDS p=0.965 OK
  lag=28d fol

## 10. Tabla comparativa y tablas por fold

In [13]:
comp_rows = []
for mtype in ['MLP', 'RNN', 'LSTM']:
    for lag in LAGS_LIST:
        folds     = list(results_all[mtype][lag].keys())
        rmse_list = [results_all[mtype][lag][f]['rmse_test'] for f in folds]
        mape_list = [results_all[mtype][lag][f]['metrics_test']['mean']['MAPE'] for f in folds]
        bds_list  = [results_all[mtype][lag][f]['bds_pval'] for f in folds]
        comp_rows.append({
            'Modelo':        mtype,
            'Lag (dias)':    lag,
            'RMSE mean±std': f"{np.mean(rmse_list):.4f} +/- {np.std(rmse_list):.4f}",
            'MAPE mean±std': f"{np.mean(mape_list):.2f}% +/- {np.std(mape_list):.2f}%",
            'BDS p mean':    f"{np.mean(bds_list):.3f}",
            'BDS p>0.05':    f"{sum(1 for p in bds_list if p>0.05)}/{len(folds)}",
        })

comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv('results/summary_comparison.csv', index=False)
display(comp_df)

for mtype in ['MLP', 'RNN', 'LSTM']:
    for lag in LAGS_LIST:
        rows = []
        for fold in list(results_all[mtype][lag].keys()):
            r = results_all[mtype][lag][fold]; m = r['metrics_test']
            row = {'Fold': f'Fold {fold}'}
            for h in range(1, N_STEPS_FORECAST+1):
                row[f'MAPE_h{h}'] = m[f'h{h}']['MAPE']
                row[f'RMSE_h{h}'] = m[f'h{h}']['RMSE']
            row['MAPE_mean'] = m['mean']['MAPE']; row['MAE_mean']  = m['mean']['MAE']
            row['RMSE_mean'] = m['mean']['RMSE']; row['MSE_mean']  = m['mean']['MSE']
            row['BDS_pval_h1'] = r['bds_pval']
            rows.append(row)
        df_t = pd.DataFrame(rows).set_index('Fold')
        means = df_t.mean().round(6); stds = df_t.std().round(6)
        df_t.loc['Mean +/- Std'] = {col: f"{means[col]:.4f} +/- {stds[col]:.4f}" for col in df_t.columns}
        df_t.to_csv(f'results/metrics_{mtype}_lag{lag}.csv')

print("\n✅ Tablas guardadas en results/")


,Modelo,Lag (dias),RMSE mean±std,MAPE mean±std,BDS p mean,BDS p>0.05
0,MLP,7,0.2483 +/- 0.0128,41.80% +/- 1.25%,0.959,5/5
1,MLP,14,0.2245 +/- 0.0275,45.34% +/- 3.79%,0.964,5/5
2,MLP,21,0.2323 +/- 0.0208,41.39% +/- 1.87%,0.948,5/5
3,MLP,28,0.3786 +/- 0.0825,51.92% +/- 8.67%,0.971,5/5
4,RNN,7,0.2511 +/- 0.0111,42.14% +/- 1.19%,0.945,5/5
5,RNN,14,0.2339 +/- 0.0233,44.75% +/- 4.37%,0.977,5/5
6,RNN,21,0.2449 +/- 0.0160,45.23% +/- 1.09%,0.968,5/5
7,RNN,28,0.3941 +/- 0.0868,51.65% +/- 5.62%,0.959,5/5
8,LSTM,7,0.2475 +/- 0.0115,43.00% +/- 1.07%,0.942,5/5
9,LSTM,14,0.2200 +/- 0.0252,43.78% +/- 1.12%,0.952,5/5



✅ Tablas guardadas en results/


## 11. Guardar mejor modelo

In [15]:
import contextlib

best_global = {'rmse': np.inf, 'mtype': None, 'lag': None, 'fold': None}
for mtype in ['MLP', 'RNN', 'LSTM']:
    for lag in LAGS_LIST:
        for fold, r in results_all[mtype][lag].items():
            if r['rmse_test'] < best_global['rmse']:
                best_global = {'rmse': r['rmse_test'], 'mtype': mtype, 'lag': lag, 'fold': fold}

print(f"Mejor modelo: {best_global['mtype']} | lag={best_global['lag']}d | "
      f"fold={best_global['fold']} | RMSE={best_global['rmse']:.4f}")

bm = best_global
sp = splits[bm['lag']]
arch_b  = best_configs[bm['mtype']]['arquitectura']
dmask_b = best_configs[bm['mtype']]['dropout_mask']

X_tr_r  = sp['X'][bm['fold']];   y_tr_r  = sp['y'][bm['fold']]
X_val_r = sp['Xcv'][bm['fold']]; y_val_r = sp['ycv'][bm['fold']]
sx = StandardScaler().fit(X_tr_r); sy = StandardScaler().fit(y_tr_r)

if bm['mtype'] in ('RNN', 'LSTM'):
    X_in     = sx.transform(X_tr_r).reshape(-1, bm['lag'], 1)
    X_val_in = sx.transform(X_val_r).reshape(-1, bm['lag'], 1)
else:
    X_in     = sx.transform(X_tr_r)
    X_val_in = sx.transform(X_val_r)

tf.keras.backend.clear_session(); tf.random.set_seed(42)
if   bm['mtype'] == 'MLP':  best_model = build_mlp(arch_b, dmask_b, bm['lag'], N_STEPS_FORECAST)
elif bm['mtype'] == 'RNN':  best_model = build_rnn(arch_b, dmask_b, bm['lag'], N_STEPS_FORECAST)
else:                        best_model = build_lstm(arch_b, dmask_b, bm['lag'], N_STEPS_FORECAST)

ctx = tf.device('/CPU:0') if bm['mtype'] == 'LSTM' else contextlib.nullcontext()
with ctx:
    best_model.fit(X_in, sy.transform(y_tr_r),
                   validation_data=(X_val_in, sy.transform(y_val_r)),
                   epochs=EPOCHS, batch_size=BATCH_SIZE,
                   callbacks=[tf.keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True)],
                   verbose=0)

best_model.save('app/model_keras.keras')
joblib.dump({'scaler_x': sx, 'scaler_y': sy, 'lag': bm['lag'],
             'model_type': bm['mtype'], 'best_global': best_global}, 'app/scalers.joblib')

with open('results.pkl', 'wb') as f:
    pickle.dump({
        'results_all':    results_all,
        'best_configs':   best_configs,
        'best_global':    best_global,
        'lags_list':      LAGS_LIST,
        'n_steps_forecast': N_STEPS_FORECAST,
        'target_col':     TARGET_COL,
        'time_series':    time_series,
        'dates':          dates,
    }, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"\n✅ app/model_keras.keras guardado")
print(f"✅ app/scalers.joblib guardado")
print(f"✅ results.pkl guardado")

Mejor modelo: LSTM | lag=21d | fold=4 | RMSE=0.1896

✅ app/model_keras.keras guardado
✅ app/scalers.joblib guardado
✅ results.pkl guardado
